# Control Points & Pixel Geolocation

Two things live in this notebook:
1. **Real-world location** of the 5 main MODIS pixels (moved here from
   the temporal-overlap notebook, since it's spatial, not temporal,
   content)
2. **Control-point correlation** -- testing whether the main HF365<->MODIS
   correlation reflects a real relationship or a coincidence, by
   checking correlation at locations of increasing distance/dissimilarity

## Bug fix in this version
The chart-plotting loop previously used a hardcoded list of location
names with underscores (`MA_Far`, `Japan_Opposite_TZ`, ...), but the
actual `ID` values returned by AppEEARS came back **without**
underscores (`MAFar`, `JapanOppositeTZ`, ...) -- causing a `KeyError`
that stopped every chart in this notebook from ever being generated.
**Fixed by reading the location list directly from the data** (`raw[id_col].unique()`) instead of a separate hardcoded list, so the two can never
drift out of sync again.


In [ ]:
import os
import re
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import rasterio
from numpy.polynomial import polynomial as P

sns.set_style("whitegrid")

# vvv EDIT THESE to match your folders/files vvv
HF365_DIR = "."
MODIS_DIR = "modis_downloads"              # the 5 main pixels (for geolocation)
CONTROL_CSV = "ControlPoints2-MCD15A2H-061-results.csv"
# ^^^ EDIT THESE to match your folders/files vvv

HF365_FILES = {
    "EVI2": "hf365-01-hls-evi2.csv",
    "NDVI": "hf365-02-hls-ndvi.csv",
    "NIRv": "hf365-03-hls-nirv.csv",
}
MODIS_VARS = ["Lai_500m", "Fpar_500m"]


## Part 1: Real-world location of the 5 main MODIS pixels

Reads each file's embedded CRS + affine transform and reprojects the
real `[1,1]` cell to WGS84 lat/lon.

In [ ]:
from rasterio.warp import transform as rio_transform

fpar_files = glob.glob(os.path.join(MODIS_DIR, "**", "*Fpar_500m*.tif"), recursive=True)
pixel_pattern = re.compile(r"aid(\d+)\.tif$")

pixel_coords = {}
for path in fpar_files:
    m = pixel_pattern.search(os.path.basename(path))
    if not m:
        continue
    pixel_id = f"pixel_{int(m.group(1))}"
    if pixel_id in pixel_coords:
        continue
    with rasterio.open(path) as src:
        lon_native, lat_native = src.xy(1, 1)
        lon_wgs84, lat_wgs84 = rio_transform(src.crs, "EPSG:4326", [lon_native], [lat_native])
    pixel_coords[pixel_id] = (lat_wgs84[0], lon_wgs84[0])

coords_df = pd.DataFrame(pixel_coords, index=["latitude", "longitude"]).T.sort_index()
print("MODIS pixel locations (WGS84 lat/lon):")
coords_df


## Part 2: HF365 plots vs 4 control points -- full correlation breakdown

- **MAFar** -- Mount Greylock area, western MA (same state, ~130km away)
- **DifferentState** -- Wind River Experimental Forest, WA
- **JapanOppositeTZ** -- Takayama, Japan (~13-14hr timezone offset)
- **AmazonEquatorial** -- Tapajos National Forest, Brazil (near-equatorial, weak seasonality)

Same "4 views x 6 index-pairs" structure as the main correlation
analysis. Since each control point is a single pixel (not 5), views
[1]/[3] and [2]/[4] are mathematically identical by construction --
noted rather than hidden.

In [ ]:
hf365_wide = {}
for name, fname in HF365_FILES.items():
    df = pd.read_csv(os.path.join(HF365_DIR, fname))
    df.columns = [c.strip() for c in df.columns]
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col).sort_index()
    hf365_wide[name] = df
    print(f"{name}: {df.shape[0]} dates x {df.shape[1]} plots")


In [ ]:
raw = pd.read_csv(CONTROL_CSV)
raw["Date"] = pd.to_datetime(raw["Date"])
id_col = "ID" if "ID" in raw.columns else "Category"

# FIX: read locations directly from the data, never a hardcoded list
locations = sorted(raw[id_col].unique())
print(f"Using '{id_col}' as location identifier. Locations found: {locations}")

lai_col = [c for c in raw.columns if c.endswith("Lai_500m")][0]
fpar_col = [c for c in raw.columns if c.endswith("Fpar_500m")][0]
raw.loc[~raw[lai_col].between(0, 10), lai_col] = np.nan
raw.loc[~raw[fpar_col].between(0, 1), fpar_col] = np.nan

control_series = {}
for var, col in [("Lai_500m", lai_col), ("Fpar_500m", fpar_col)]:
    control_series[var] = {loc: raw[raw[id_col]==loc].set_index("Date")[col].sort_index() for loc in locations}
    for loc, s in control_series[var].items():
        print(f"{var} -- {loc}: {s.notna().sum()} valid dates out of {len(s)}")


In [ ]:
def build_windows(dates_index):
    d = dates_index.sort_values()
    return [(d[i], d[i+1] - pd.Timedelta(days=1)) for i in range(len(d)-1)]

def align_window_containment_single(hf_df, pixel_series, windows):
    matched_hf, matched_px, matched_dates = [], [], []
    for w_start, w_end in windows:
        in_window = hf_df.loc[(hf_df.index >= w_start) & (hf_df.index <= w_end)]
        if in_window.empty or w_start not in pixel_series.index:
            continue
        px_val = pixel_series.loc[w_start]
        if pd.isna(px_val):
            continue
        matched_hf.append(in_window.mean(axis=0))
        matched_px.append(px_val)
        matched_dates.append(w_start)
    if not matched_dates:
        return pd.DataFrame(columns=hf_df.columns), pd.Series(dtype=float)
    return pd.DataFrame(matched_hf, index=matched_dates), pd.Series(matched_px, index=matched_dates)

def score_views(hf_aligned, px_aligned_as_frame):
    corr_matrix = pd.DataFrame(index=hf_aligned.columns, columns=px_aligned_as_frame.columns, dtype=float)
    for plot in hf_aligned.columns:
        for pixel in px_aligned_as_frame.columns:
            corr_matrix.loc[plot, pixel] = hf_aligned[plot].corr(px_aligned_as_frame[pixel])
    avg_plot = hf_aligned.mean(axis=1)
    avg_pixel = px_aligned_as_frame.mean(axis=1)
    return {
        "plot_vs_pixel_matrix": corr_matrix,
        "avg_plot_vs_pixel": px_aligned_as_frame.apply(lambda col: avg_plot.corr(col)),
        "plot_vs_avg_pixel": hf_aligned.apply(lambda col: col.corr(avg_pixel)),
        "avg_plot_vs_avg_pixel": avg_plot.corr(avg_pixel),
    }


In [ ]:
all_results = {}
all_aligned = {}
for loc in locations:
    for hf_name, hf_df in hf365_wide.items():
        for modis_var in MODIS_VARS:
            pixel_series = control_series[modis_var][loc]
            windows = build_windows(pixel_series.index)
            hf_a, px_a_series = align_window_containment_single(hf_df, pixel_series, windows)
            px_a_frame = px_a_series.to_frame(name=loc)
            key = (loc, hf_name, modis_var)
            all_results[key] = score_views(hf_a, px_a_frame) if len(hf_a) >= 2 else None
            all_aligned[key] = (hf_a, px_a_frame)

print(f"Total combinations run: {len(all_results)} (expect {len(locations)} x 3 x 2 = {len(locations)*6})")


## Summary table -- all combinations

In [ ]:
summary_rows = []
for (loc, hf_name, modis_var), res in all_results.items():
    hf_a, _ = all_aligned[(loc, hf_name, modis_var)]
    summary_rows.append(dict(
        location=loc, hf_index=hf_name, modis_var=modis_var,
        avg_plot_vs_avg_pixel=res["avg_plot_vs_avg_pixel"] if res else np.nan,
        n_matched=len(hf_a),
    ))
summary_df = pd.DataFrame(summary_rows).round(3)
summary_df


## All four-panel charts (fix applied: loops over real location names)

In [ ]:
def plot_four_views(loc, hf_name, modis_var):
    key = (loc, hf_name, modis_var)
    res = all_results[key]
    hf_a, px_a_frame = all_aligned[key]
    fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))
    fig.suptitle(f"{hf_name} vs {modis_var}  --  Control point: {loc}", fontsize=12, y=1.05)

    if res is None or len(hf_a) < 2:
        for ax in axes:
            ax.text(0.5, 0.5, "Not enough matched points", ha="center", va="center", transform=ax.transAxes)
            ax.set_xticks([]); ax.set_yticks([])
        plt.tight_layout(); plt.show(); return

    pixel_col = px_a_frame.columns[0]

    ax = axes[0]
    vals1 = res["plot_vs_pixel_matrix"][pixel_col]
    ax.bar(range(len(vals1)), vals1.values, color="steelblue")
    ax.set_xticks(range(len(vals1))); ax.set_xticklabels(vals1.index, rotation=90, fontsize=5)
    ax.set_ylim(-1, 1); ax.set_title("[1] Each Plot vs Pixel")

    ax = axes[1]
    ax.bar([pixel_col], [res["avg_plot_vs_pixel"][pixel_col]], color="darkorange")
    ax.set_ylim(-1, 1); ax.set_title(f"[2] Avg-plot vs Pixel (r={res['avg_plot_vs_pixel'][pixel_col]:.3f})")

    ax = axes[2]
    vals3 = res["plot_vs_avg_pixel"]
    ax.bar(range(len(vals3)), vals3.values, color="seagreen")
    ax.set_xticks(range(len(vals3))); ax.set_xticklabels(vals3.index, rotation=90, fontsize=5)
    ax.set_ylim(-1, 1); ax.set_title("[3] Each Plot vs \"Avg Pixel\" (== [1] here)")

    ax = axes[3]
    avg_plot = hf_a.mean(axis=1)
    pixel_series = px_a_frame[pixel_col]
    ax.scatter(avg_plot, pixel_series, s=18, alpha=0.6, color="purple")
    mask = avg_plot.notna() & pixel_series.notna()
    if mask.sum() >= 2:
        coefs = P.polyfit(avg_plot[mask], pixel_series[mask], 1)
        xs = np.linspace(avg_plot[mask].min(), avg_plot[mask].max(), 50)
        ax.plot(xs, P.polyval(xs, coefs), color="black", linewidth=1)
    ax.set_title(f"[4] Avg-plot vs Avg-pixel (r={res['avg_plot_vs_avg_pixel']:.3f}, == [2])")
    plt.tight_layout(); plt.show()

INDEX_PAIRS = [(hf, mv) for hf in HF365_FILES for mv in MODIS_VARS]


In [ ]:
# FIX: iterate over `locations` (read from the data) instead of a hardcoded list
for loc in locations:
    for hf_name, modis_var in INDEX_PAIRS:
        plot_four_views(loc, hf_name, modis_var)


## Side-by-side summary chart -- all control points at once

In [ ]:
plot_df = summary_df.copy()
plot_df["pair"] = plot_df["hf_index"] + " vs " + plot_df["modis_var"]

fig, ax = plt.subplots(figsize=(13, 6))
sns.barplot(data=plot_df, x="pair", y="avg_plot_vs_avg_pixel", hue="location", ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylim(-1, 1)
ax.set_ylabel("Avg-plot vs pixel correlation (r)")
ax.set_xlabel("")
ax.set_title("All control points, all 6 index-pairs")
plt.xticks(rotation=20, ha="right")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
